# 13 — Calibration, bootstrap CIs & McNemar

**Purpose:** Quantify how much we should trust the model's confidence + whether the 0.750 vs 0.765 model selection was a real difference.

Covers AM2 criteria **K20** (sources of error), **S22/S24 distinction** (robust monitoring), and **strengthens the S9/K14 distinction story** — McNemar will likely show the SBERT no-meta vs with-meta F1 difference (0.750 vs 0.765) is statistical noise, which means the SHAP-driven decision didn't trade real F1 for principle.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import f1_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
MODEL_DIR = ROOT / "models" / "runs" / "v1_2026-05-16"

with open(MODEL_DIR / "run_metadata.json") as f:
    run_meta = json.load(f)

VAL_CSV = ROOT / "data" / "modelling" / "val.csv"
val_df = pd.read_csv(VAL_CSV)
print(f"Val set: {len(val_df)} items")


In [ ]:
# Encode val titles + predict with production model.
encoder = SentenceTransformer(run_meta['embedding_model'])
classifier = joblib.load(MODEL_DIR / "classifier.joblib")

title_col = 'title' if 'title' in val_df.columns else 'text'
target_col = 'target' if 'target' in val_df.columns else 'theme'
y_true = val_df[target_col].astype(str).values

X = encoder.encode(val_df[title_col].fillna('').astype(str).tolist(), show_progress_bar=True)
y_pred = classifier.predict(X)
y_proba = classifier.predict_proba(X)
classes = classifier.classes_
print(f"Predictions: {len(y_pred)}")


## Part 1 — Reliability diagram (calibration)

In [ ]:
# Per-class reliability diagram.
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, cls in zip(axes.flat, classes):
    cls_idx = list(classes).index(cls)
    y_true_binary = (y_true == cls).astype(int)
    y_pred_proba = y_proba[:, cls_idx]
    try:
        prob_true, prob_pred = calibration_curve(y_true_binary, y_pred_proba, n_bins=8, strategy='uniform')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Ideal')
        ax.plot(prob_pred, prob_true, marker='o', label=cls)
        ax.set_xlabel('Predicted probability')
        ax.set_ylabel('Actual frequency')
        ax.set_title(f'Reliability: {cls}')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, f'Not enough data\n({e})', ha='center', va='center')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'calibration_reliability.png', dpi=120)
plt.show()


## Part 2 — Brier score & Expected Calibration Error

In [ ]:
# Brier score per class.
print('=== Brier score per class (lower is better) ===')
for cls in classes:
    cls_idx = list(classes).index(cls)
    y_bin = (y_true == cls).astype(int)
    bs = brier_score_loss(y_bin, y_proba[:, cls_idx])
    print(f"  {cls:50s} {bs:.4f}")

# Expected Calibration Error — overall.
def ece(y_true_class, y_proba_class, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece_val = 0
    n = len(y_proba_class)
    for i in range(n_bins):
        mask = (y_proba_class >= bins[i]) & (y_proba_class < bins[i+1])
        if mask.sum() > 0:
            bin_acc = y_true_class[mask].mean()
            bin_conf = y_proba_class[mask].mean()
            ece_val += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return ece_val

# Multiclass ECE on max-prob class.
y_pred_max_proba = y_proba.max(axis=1)
y_pred_correct = (y_pred == y_true).astype(int)
ece_val = ece(y_pred_correct, y_pred_max_proba, n_bins=10)
print(f"\nOverall ECE (n_bins=10): {ece_val:.4f}")
print("(0 = perfectly calibrated; ≤0.05 is typically considered 'well calibrated')")


## Part 3 — Bootstrap confidence intervals on macro F1

In [ ]:
# Bootstrap 1000 resamples for 95% CI on val macro F1.
N_BOOT = 1000
rng = np.random.RandomState(42)
f1_samples = []
for _ in range(N_BOOT):
    idx = rng.choice(len(y_true), size=len(y_true), replace=True)
    f1_samples.append(f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0))

f1_samples = np.array(f1_samples)
ci_lo, ci_hi = np.percentile(f1_samples, [2.5, 97.5])
print(f"Val macro F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.3f}")
print(f"95% bootstrap CI: [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"Width: {ci_hi - ci_lo:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(f1_samples, bins=40, alpha=0.7, color='#3498db')
ax.axvline(ci_lo, color='red', linestyle='--', alpha=0.7, label=f'95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axvline(ci_hi, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Macro F1')
ax.set_ylabel('Bootstrap iterations')
ax.set_title(f'Bootstrap distribution of val macro F1 (N={N_BOOT})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'calibration_bootstrap_f1.png', dpi=120)
plt.show()


## Part 4 — Per-class bootstrap F1

In [ ]:
# Per-class CIs — narrower CIs = more confident in the class result.
print('=== Per-class 95% CI for F1 (bootstrap N=500) ===')
for cls in classes:
    samples = []
    for _ in range(500):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        cm_f1 = f1_score(y_true[idx], y_pred[idx], labels=[cls], average='macro', zero_division=0)
        samples.append(cm_f1)
    lo, hi = np.percentile(samples, [2.5, 97.5])
    print(f"  {cls:50s} [{lo:.3f}, {hi:.3f}]  width={hi-lo:.3f}")


## Part 5 — McNemar's test: SBERT no-meta vs with-meta

In [ ]:
# Compare the production (no-meta) model vs the with-meta variant.
# This is the 0.750 vs 0.765 model selection from nb 09's SHAP analysis.
# Goal: show the F1 difference is NOT statistically significant — which
# strengthens the case for shipping the no-meta model on principle.

# TODO — load the with-meta model predictions.
# Either:
#   (a) Re-train the with-meta variant here (requires train.csv + meta one-hots)
#   (b) Save with-meta predictions as a separate joblib and load
# For now, sketching the structure.

try:
    from statsmodels.stats.contingency_tables import mcnemar
    # WITH_META_MODEL = joblib.load("path-to-with-meta-classifier.joblib")
    # y_pred_with_meta = WITH_META_MODEL.predict(X_with_meta)
    #
    # Build 2x2 contingency table:
    # n11 = both correct, n10 = no-meta correct only, n01 = with-meta correct only, n00 = both wrong
    # mcnemar_result = mcnemar([[n11, n10], [n01, n00]], exact=False)
    # print(f"McNemar's test p-value: {mcnemar_result.pvalue:.4f}")
    print("TODO: load with-meta predictions and run McNemar (see comments above)")
except ImportError:
    print("statsmodels not installed — pip install statsmodels")


## Conclusions

**Calibration:** the reliability diagrams + ECE tell us whether the model's stated confidence matches its actual accuracy. Downstream uses of confidence (`scoring.py` cluster threshold ≥ 0.85, hybrid routing thresholds) only make sense if the model is reasonably well-calibrated.

**Bootstrap CIs:** the headline val macro F1 of 0.750 has a 95% CI of roughly [`ci_lo`, `ci_hi`] — wider CIs mean we should be less confident in pp differences between models. Critical for honest reporting.

**McNemar (when populated):** if the 0.750 vs 0.765 difference is not statistically significant (likely), that **strengthens the SHAP-driven decision** to ship no-meta — you didn't trade real F1 for interpretability, you traded noise for interpretability. This is the distinction-level S9/K14 evidence.